In [280]:
# Test the fixed implied_volatility function
import sys
sys.path.append('../src')

from pricing.greeks import implied_volatility

# Test case that was failing: Deep OTM put near expiry
print("Testing deep OTM put near expiry...")
print("=" * 60)

result = implied_volatility(
    option_price=0.34,
    S=321.08,
    K=263.0,
    T=0.0027,  # ~1 day to expiry
    r=0.02,
    option_type='put'
)

print(f"✓ Solved IV: {result:.4f} ({result*100:.2f}%)")
print(f"  This means the market is pricing in ~{result*100:.1f}% annualized vol")
print(f"  for a put that is {((321.08/263.0 - 1)*100):.1f}% OTM with 1 day to expiry")
print("\nNote: Very high IV is expected for deep OTM options near expiry!")
print("=" * 60)


Testing deep OTM put near expiry...
✓ Solved IV: 2.0246 (202.46%)
  This means the market is pricing in ~202.5% annualized vol
  for a put that is 22.1% OTM with 1 day to expiry

Note: Very high IV is expected for deep OTM options near expiry!


In [76]:
import pandas as pd

# Load the data
df = pd.read_parquet('../data/processed/spy_options.parquet')

# Check unique dates
unique_dates = df['date'].unique()
print(f"Number of unique dates: {len(unique_dates)}")
print(f"\nFirst 10 dates:")
print(sorted(unique_dates)[:10])

Number of unique dates: 30

First 10 dates:
['2019-06-08', '2019-06-15', '2019-06-22', '2019-06-30', '2019-07-06', '2019-07-13', '2019-07-20', '2019-07-27', '2019-08-03', '2019-08-10']


In [90]:
import numpy as np
from scipy.stats import norm
from typing import Union, Literal
def black_scholes_price(
    S: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    option_type: Literal['call', 'put'] = 'call'
) -> float:
    """
    Calculate European option price using Black-Scholes model.
    
    Args:
        S: Current underlying price
        K: Strike price
        T: Time to expiration in years (e.g., 30 days = 30/365)
        r: Risk-free rate (annualized, as decimal, e.g., 0.02 for 2%)
        sigma: Implied volatility (annualized, as decimal, e.g., 0.20 for 20%)
        option_type: 'call' or 'put'
        
    Returns:
        Theoretical option price
        
    Example:
        >>> price = black_scholes_price(S=100, K=100, T=1.0, r=0.05, sigma=0.20)
        >>> print(f"Call price: ${price:.2f}")
        Call price: $10.45
        
    TODO: Implement Black-Scholes formula
    
    Hints:
        1. Handle edge case: T = 0 (expiration) → return intrinsic value
        2. Handle edge case: sigma = 0 → return discounted intrinsic value
        3. Calculate d1 = [ln(S/K) + (r + sigma²/2)*T] / (sigma*sqrt(T))
        4. Calculate d2 = d1 - sigma*sqrt(T)
        5. Use scipy.stats.norm.cdf() for N(d1) and N(d2)
        6. Calculate call = S*N(d1) - K*exp(-r*T)*N(d2)
        7. For put, use put-call parity or put formula directly
        
    Notes:
        - All inputs should be positive (except r can be negative in rare cases)
        - T is in YEARS (divide days by 365)
        - sigma and r are ANNUALIZED rates
    """
    # Input validation
    if S <= 0:
        raise ValueError(f"Underlying price must be positive, got {S}")
    if K <= 0:
        raise ValueError(f"Strike price must be positive, got {K}")
    if T < 0:
        raise ValueError(f"Time to expiration cannot be negative, got {T}")
    if sigma < 0:
        raise ValueError(f"Volatility cannot be negative, got {sigma}")
    if option_type not in ['call', 'put']:
        raise ValueError(f"option_type must be 'call' or 'put', got {option_type}")

    # Calculate d1 and d2
    denom = sigma* np.sqrt(T)
    d_1_num = np.log(S / K) + (r + (sigma ** 2) / 2) * T
    d1 = d_1_num / denom if denom != 0 else 0
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'call':
        # If contract expired, return 0 or difference between underlying and strike
        if T == 0:
            return intrinsic_value(S, K, option_type)
        
        # If no vol, return discounted intrinsic value
        if sigma == 0:
            return max(S - K * np.exp(-r*T), 0)

        return (S * norm.cdf(d1)) - (K * np.exp(-r*T) * norm.cdf(d2))

    if option_type == 'put':
        # If contract expired, return 0 or difference between underlying and strike
        if T == 0:
            return intrinsic_value(S, K, option_type)

        # If no vol, return discounted intrinsic value
        if sigma == 0:
            return max((K * np.exp(-r*T)) - S, 0)

        return (K * np.exp(-r*T) * norm.cdf(-d2)) - (S * norm.cdf(-d1)) 
     

print(black_scholes_price(S=287.6499938964844,K=293, T=13/365, r=0.02, sigma=0.1521, option_type='put'))


6.507575496022497


In [ ]:
"strike":293,"call_put":"Put","bid":7,"ask":7.19,"vol":0.1521,"days_to_expiry":13,"underlying_price":287.6499938964844,risk_free_rate":0.02


In [ ]:
# Load the processed data
df = pd.read_parquet('../data/processed/spy_options.parquet')

# Look at the structure
print("Columns:", df.columns.tolist())
print(f"\nShape: {df.shape}")
print("\nFirst few rows:")
df.head()


In [79]:
# Pick a single date for testing
test_date = '2019-06-08'
df_date = df[df['date'] == test_date].copy()

print(f"Options on {test_date}: {len(df_date)}")
print(f"\nUnderlying price on this date: ${df_date['underlying_price'].iloc[0]:.2f}")
print(f"Risk-free rate: {df_date['risk_free_rate'].iloc[0]:.4f}")

# Show the distribution of options
print(f"\nBreakdown:")
print(f"  Calls: {(df_date['call_put'] == 'Call').sum()}")
print(f"  Puts: {(df_date['call_put'] == 'Put').sum()}")
print(f"\nDays to expiry range: {df_date['days_to_expiry'].min()} to {df_date['days_to_expiry'].max()}")


Options on 2019-06-08: 55

Underlying price on this date: $287.65
Risk-free rate: 0.0200

Breakdown:
  Calls: 25
  Puts: 30

Days to expiry range: 13 to 48


In [80]:
# Filter for 30-day options (closest to 30 DTE)
df_30day = df_date[df_date['days_to_expiry'].between(25, 35)].copy()

print(f"30-day options: {len(df_30day)}")

# Get underlying price
S = df_30day['underlying_price'].iloc[0]

# Calculate moneyness for each option
df_30day['moneyness'] = df_30day['strike'] / S

# Separate calls and puts
calls = df_30day[df_30day['call_put'] == 'Call'].copy()
puts = df_30day[df_30day['call_put'] == 'Put'].copy()

# Find ATM, ITM, OTM for each
def get_representative_options(options_df, S, option_type):
    """Get ATM, ITM, OTM examples."""
    options_df = options_df.copy()
    options_df['dist_from_atm'] = abs(options_df['strike'] - S)
    
    # ATM - closest to current price
    atm = options_df.loc[options_df['dist_from_atm'].idxmin()]
    
    if option_type == 'call':
        # ITM call: strike < S
        itm_options = options_df[options_df['strike'] < S]
        itm = itm_options.loc[itm_options['dist_from_atm'].idxmin()] if len(itm_options) > 0 else None
        
        # OTM call: strike > S
        otm_options = options_df[options_df['strike'] > S]
        otm = otm_options.loc[otm_options['dist_from_atm'].idxmin()] if len(otm_options) > 0 else None
    else:  # put
        # ITM put: strike > S
        itm_options = options_df[options_df['strike'] > S]
        itm = itm_options.loc[itm_options['dist_from_atm'].idxmin()] if len(itm_options) > 0 else None
        
        # OTM put: strike < S
        otm_options = options_df[options_df['strike'] < S]
        otm = otm_options.loc[otm_options['dist_from_atm'].idxmin()] if len(otm_options) > 0 else None
    
    return {'ATM': atm, 'ITM': itm, 'OTM': otm}

# Get examples
call_examples = get_representative_options(calls, S, 'call')
put_examples = get_representative_options(puts, S, 'put')

print(f"\n{'='*80}")
print(f"UNDERLYING: SPY = ${S:.2f}")
print(f"{'='*80}")


30-day options: 19

UNDERLYING: SPY = $287.65


In [81]:
# Import the pricer from your src folder
import sys
sys.path.append('..')
from src.pricer import black_scholes_price

def test_option(option_row):
    """Test Black-Scholes pricer against market price."""
    # Extract parameters
    S = option_row['underlying_price']
    K = option_row['strike']
    T = option_row['days_to_expiry'] / 365.0  # Convert to years
    r = option_row['risk_free_rate']
    sigma = option_row['vol']
    option_type = option_row['call_put'].lower()
    
    # Market price (mid of bid-ask)
    market_price = option_row['mid_price']
    
    # Theoretical price
    bs_price = black_scholes_price(S, K, T, r, sigma, option_type)
    
    # Calculate error
    abs_error = bs_price - market_price
    pct_error = (abs_error / market_price) * 100 if market_price > 0 else 0
    
    return {
        'Market Price': market_price,
        'BS Price': bs_price,
        'Abs Error': abs_error,
        'Pct Error': pct_error
    }

# Test all examples
results = []

print("\n" + "="*80)
print("CALL OPTIONS")
print("="*80)

for moneyness_type, option in call_examples.items():
    if option is not None:
        result = test_option(option)
        
        print(f"\n{moneyness_type} CALL (Strike: ${option['strike']:.2f})")
        print(f"  Market Price:  ${result['Market Price']:8.2f}")
        print(f"  BS Price:      ${result['BS Price']:8.2f}")
        print(f"  Error:         ${result['Abs Error']:8.2f} ({result['Pct Error']:+.1f}%)")
        print(f"  IV: {option['vol']:.2%}, DTE: {option['days_to_expiry']} days")
        
        results.append({
            'Type': f"Call {moneyness_type}",
            'Strike': option['strike'],
            'Market': result['Market Price'],
            'BS': result['BS Price'],
            'Error': result['Abs Error'],
            'Error %': result['Pct Error']
        })

print("\n" + "="*80)
print("PUT OPTIONS")
print("="*80)

for moneyness_type, option in put_examples.items():
    if option is not None:
        result = test_option(option)
        
        print(f"\n{moneyness_type} PUT (Strike: ${option['strike']:.2f})")
        print(f"  Market Price:  ${result['Market Price']:8.2f}")
        print(f"  BS Price:      ${result['BS Price']:8.2f}")
        print(f"  Error:         ${result['Abs Error']:8.2f} ({result['Pct Error']:+.1f}%)")
        print(f"  IV: {option['vol']:.2%}, DTE: {option['days_to_expiry']} days")
        
        results.append({
            'Type': f"Put {moneyness_type}",
            'Strike': option['strike'],
            'Market': result['Market Price'],
            'BS': result['BS Price'],
            'Error': result['Abs Error'],
            'Error %': result['Pct Error']
        })

# Summary
results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(results_df.to_string(index=False))
print(f"\nMean Absolute Error: ${results_df['Error'].abs().mean():.2f}")
print(f"Mean Percentage Error: {results_df['Error %'].abs().mean():.1f}%")



CALL OPTIONS

ATM CALL (Strike: $287.50)
  Market Price:  $    4.29
  BS Price:      $    4.80
  Error:         $    0.51 (+11.9%)
  IV: 14.45%, DTE: 27 days

ITM CALL (Strike: $287.50)
  Market Price:  $    4.29
  BS Price:      $    4.80
  Error:         $    0.51 (+11.9%)
  IV: 14.45%, DTE: 27 days

OTM CALL (Strike: $293.00)
  Market Price:  $    1.76
  BS Price:      $    2.05
  Error:         $    0.29 (+16.3%)
  IV: 12.83%, DTE: 27 days

PUT OPTIONS

ATM PUT (Strike: $287.50)
  Market Price:  $    4.86
  BS Price:      $    4.20
  Error:         $   -0.66 (-13.5%)
  IV: 14.39%, DTE: 27 days

ITM PUT (Strike: $293.00)
  Market Price:  $    7.92
  BS Price:      $    7.06
  Error:         $   -0.87 (-11.0%)
  IV: 13.16%, DTE: 27 days

OTM PUT (Strike: $287.50)
  Market Price:  $    4.86
  BS Price:      $    4.20
  Error:         $   -0.66 (-13.5%)
  IV: 14.39%, DTE: 27 days

SUMMARY
    Type  Strike  Market       BS     Error    Error %
Call ATM   287.5   4.290 4.798563  0.50856

In [ ]:
# Extract the raw examples for documentation/manual testing
print("\n" + "="*80)
print("RAW EXAMPLES FOR MANUAL TESTING")
print("="*80)

all_examples = [
    ('ATM Call', call_examples['ATM']),
    ('ITM Call', call_examples['ITM']),
    ('OTM Call', call_examples['OTM']),
    ('ATM Put', put_examples['ATM']),
    ('ITM Put', put_examples['ITM']),
    ('OTM Put', put_examples['OTM']),
]

for name, option in all_examples:
    if option is not None:
        print(f"\n{name}:")
        print(f"  black_scholes_price(")
        print(f"    S={option['underlying_price']:.2f},")
        print(f"    K={option['strike']:.2f},")
        print(f"    T={option['days_to_expiry']}/365,  # {option['days_to_expiry']} days")
        print(f"    r={option['risk_free_rate']:.4f},")
        print(f"    sigma={option['vol']:.4f},")
        print(f"    option_type='{option['call_put']}'")
        print(f"  )")
        print(f"  # Expected (market price): ${option['mid_price']:.2f}")



RAW EXAMPLES FOR MANUAL TESTING

ATM Call:
  black_scholes_price(
    S=260.60,
    K=259.00,
    T=27/365,  # 27 days
    r=0.0200,
    sigma=0.3082,
    option_type='Call'
  )
  # Expected (market price): $29.16

ITM Call:
  black_scholes_price(
    S=260.60,
    K=259.00,
    T=27/365,  # 27 days
    r=0.0200,
    sigma=0.3082,
    option_type='Call'
  )
  # Expected (market price): $29.16

OTM Call:
  black_scholes_price(
    S=260.60,
    K=267.50,
    T=27/365,  # 27 days
    r=0.0200,
    sigma=0.2481,
    option_type='Call'
  )
  # Expected (market price): $20.90

ATM Put:
  black_scholes_price(
    S=260.60,
    K=259.00,
    T=27/365,  # 27 days
    r=0.0200,
    sigma=0.2286,
    option_type='Put'
  )
  # Expected (market price): $0.39

ITM Put:
  black_scholes_price(
    S=260.60,
    K=267.50,
    T=27/365,  # 27 days
    r=0.0200,
    sigma=0.2028,
    option_type='Put'
  )
  # Expected (market price): $0.79

OTM Put:
  black_scholes_price(
    S=260.60,
    K=259.00,
  

In [283]:
def intrinsic_value(
    S: float,
    K: float,
    option_type: Literal['call', 'put'] = 'call'
) -> float:
    """
    Calculate intrinsic value of an option.
    
    Args:
        S: Current underlying price
        K: Strike price
        option_type: 'call' or 'put'
        
    Returns:
        Intrinsic value (always >= 0)
        
    TODO: Implement intrinsic value calculation
    
    Hints:
        - Call intrinsic value: max(S - K, 0)
        - Put intrinsic value: max(K - S, 0)
    """
    # TODO: Implement
    if option_type == 'call':
        return max(S - K, 0)

    if option_type == 'put':
        return max(K - S, 0)

def moneyness(
    S: float,
    K: float,
    option_type: Literal['call', 'put'] = 'call'
) -> str:
    """
    Determine if option is ITM, ATM, or OTM.
    
    Args:
        S: Current underlying price
        K: Strike price
        option_type: 'call' or 'put'
        
    Returns:
        'ITM', 'ATM', or 'OTM'
        
    TODO: Implement moneyness classification
    
    Hints:
        - Use 0.5% threshold for ATM (|S-K|/S < 0.005)
        - Call ITM: S > K
        - Put ITM: K > S
    """
    # Check ATM first
    if np.abs(S - K) / S < 0.005:
        return 'ATM'
    
    # Check based on option type
    if option_type == 'call':
        return 'ITM' if S > K else 'OTM'
    else:  # put
        return 'ITM' if S < K else 'OTM'
   
def delta(
    S: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    option_type: Literal['call', 'put']
) -> float:
    """
    Calculate option delta (∂V/∂S).
    
    Delta measures how much the option price changes when the stock price 
    moves by $1. It's also interpreted as the hedge ratio - how many shares 
    to hold to delta-hedge one option.
    
    Formula:
        Call Delta = N(d1)
        Put Delta = N(d1) - 1 = -N(-d1)
    
    Where:
        d1 = [ln(S/K) + (r + σ²/2)T] / (σ√T)
        N(x) = cumulative normal distribution
    
    Args:
        S: Current stock price
        K: Strike price
        T: Time to expiration (years)
        r: Risk-free rate (annual)
        sigma: Volatility (annual)
        option_type: 'call' or 'put'
    
    Returns:
        Delta value (between 0 and 1 for calls, -1 and 0 for puts)
    
    Examples:
        >>> delta(100, 100, 1.0, 0.05, 0.20, 'call')
        0.6368...  # ATM call delta ~0.5-0.6
        
        >>> delta(100, 100, 1.0, 0.05, 0.20, 'put')
        -0.3632... # ATM put delta ~-0.4 to -0.5
    
    Edge Cases:
        - At expiration (T=0): Delta is 1 for ITM, 0 for OTM
        - Deep ITM: Call delta → 1, Put delta → -1
        - Deep OTM: Call delta → 0, Put delta → 0
    
    Interview Question: "Why is ATM delta ~0.5?"
        ATM options have ~50% probability of finishing ITM, and delta 
        approximates the probability of the option expiring in the money.
    """

    # Calculate d1
    denom = sigma* np.sqrt(T)
    d_1_num = np.log(S / K) + (r + (sigma ** 2) / 2) * T
    d1 = d_1_num / denom if denom != 0 else 0
    
    if option_type == 'call':
        if T == 0:
            return 1 if moneyness(S, K, option_type) == 'ITM' else 0
        
        return norm.cdf(d1)


    if option_type == 'put':
        if T == 0:
            return -1 if moneyness(S, K, option_type) == 'ITM' else 0
        return norm.cdf(d1) - 1

def gamma(
    S: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    option_type: Literal['call', 'put']
) -> float:
    """
    Calculate option gamma (∂²V/∂S² = ∂Δ/∂S).
    
    Gamma measures how fast delta changes as the stock price moves.
    High gamma means delta is very sensitive to price moves - dangerous 
    for hedging! Gamma is highest for ATM options near expiration.
    
    Formula:
        Gamma (same for calls and puts) = N'(d1) / (S·σ·√T)
    
    Where:
        N'(x) = standard normal PDF = (1/√2π)·e^(-x²/2)
        d1 = [ln(S/K) + (r + σ²/2)T] / (σ√T)
    
    Args:
        S: Current stock price
        K: Strike price
        T: Time to expiration (years)
        r: Risk-free rate (annual)
        sigma: Volatility (annual)
        option_type: 'call' or 'put' (doesn't matter, gamma is same)
    
    Returns:
        Gamma value (always positive)
    
    Examples:
        >>> gamma(100, 100, 1.0, 0.05, 0.20, 'call')
        0.0184...  # ATM gamma
        
        >>> gamma(100, 120, 1.0, 0.05, 0.20, 'call')
        0.0089...  # OTM gamma (lower)
    
    Edge Cases:
        - At expiration (T=0): Gamma → infinity at ATM (discontinuity)
        - Deep ITM/OTM: Gamma → 0
    
    Interview Question: "Why does gamma increase near expiration?"
        As T→0, the option becomes more binary (in or out), so delta 
        changes very rapidly near the strike. This makes ATM options 
        near expiry very difficult to hedge.
    """
    # Handle expiration cases
    if T == 0:
        return 0 if moneyness(S, K, option_type) != 'ATM' else np.inf

    # Calculate d1
    denom = sigma* np.sqrt(T)
    d_1_num = np.log(S / K) + (r + (sigma ** 2) / 2) * T
    d1 = d_1_num / denom if denom != 0 else 0
    
    return norm.pdf(d1) / (S * sigma * np.sqrt(T))

def vega(
    S: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    option_type: Literal['call', 'put']
) -> float:
    """
    Calculate option vega (∂V/∂σ).
    
    Vega measures how much the option price changes when implied volatility 
    changes by 1 percentage point (e.g., from 20% to 21%).
    
    Note: Vega is NOT actually a Greek letter (it's made up)!
    
    Formula:
        Vega (same for calls and puts) = S·√T·N'(d1)
    
    Where:
        N'(x) = standard normal PDF = (1/√2π)·e^(-x²/2)
        d1 = [ln(S/K) + (r + σ²/2)T] / (σ√T)
    
    Args:
        S: Current stock price
        K: Strike price
        T: Time to expiration (years)
        r: Risk-free rate (annual)
        sigma: Volatility (annual)
        option_type: 'call' or 'put' (doesn't matter, vega is same)
    
    Returns:
        Vega value (price change per 1% vol change)
    
    Examples:
        >>> vega(100, 100, 1.0, 0.05, 0.20, 'call')
        39.89...  # If vol goes 20% → 21%, price increases by ~$39.89
        
        >>> vega(100, 100, 0.1, 0.05, 0.20, 'call')
        12.61...  # Less time = less vega
    
    Edge Cases:
        - At expiration (T=0): Vega = 0 (no time for vol to matter)
        - ATM options: Highest vega
        - Deep ITM/OTM: Lower vega
    
    Interview Question: "Why do traders care about vega?"
        Volatility is mean-reverting. If you buy when vol is low and 
        sell when vol is high, you can profit from vega even if the 
        stock doesn't move. This is called "vol trading."
    """
    if T == 0:
        return 0

    # Calculate d1
    denom = sigma* np.sqrt(T)
    d_1_num = np.log(S / K) + (r + (sigma ** 2) / 2) * T
    d1 = d_1_num / denom if denom != 0 else 0

    return S * np.sqrt(T) * norm.pdf(d1) / 100

def rho(
    S: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    option_type: Literal['call', 'put']
) -> float:
    """
    Calculate option rho (∂V/∂r).
    
    Rho measures how much the option price changes when interest rates 
    change by 1 percentage point (e.g., from 5% to 6%).
    
    Rho is the LEAST important Greek in practice - rate changes are slow 
    and usually dominated by other factors.
    
    Formula:
        Call Rho = K·T·e^(-rT)·N(d2)
        Put Rho = -K·T·e^(-rT)·N(-d2)
    
    Where:
        d2 = d1 - σ√T
        d1 = [ln(S/K) + (r + σ²/2)T] / (σ√T)
    
    Args:
        S: Current stock price
        K: Strike price
        T: Time to expiration (years)
        r: Risk-free rate (annual)
        sigma: Volatility (annual)
        option_type: 'call' or 'put'
    
    Returns:
        Rho value (price change per 1% rate change)
    
    Examples:
        >>> rho(100, 100, 1.0, 0.05, 0.20, 'call')
        51.75...  # If rates go 5% → 6%, call gains ~$51.75
        
        >>> rho(100, 100, 1.0, 0.05, 0.20, 'put')
        -46.03... # If rates go 5% → 6%, put loses ~$46.03
    
    Edge Cases:
        - Longer maturity: Higher rho
        - At expiration: Rho = 0
        - Calls: Positive rho (benefit from higher rates)
        - Puts: Negative rho (hurt by higher rates)
    
    Interview Question: "Why do calls benefit from higher rates?"
        Higher rates reduce the present value of paying the strike price, 
        making calls more valuable. Think of it as cheaper financing.
    """
    # TODO: Implement rho calculation
    # Hint: Use np.exp(-r*T) for discounting
    # Rho is usually expressed per 1% rate change (divide by 100)
    if T == 0:
        return 0

    # Calculate d1 and d2
    denom = sigma* np.sqrt(T)
    d_1_num = np.log(S / K) + (r + (sigma ** 2) / 2) * T
    d1 = d_1_num / denom if denom != 0 else 0
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'call':
        return (K * T * np.exp(-r * T) * norm.cdf(d2)) / 100
    
    if option_type == 'put':
        return (-K * T * np.exp(-r * T) * norm.cdf(-d2)) / 100

def theta(
    S: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    option_type: Literal['call', 'put']
) -> float:
    """
    Calculate option theta (∂V/∂t).
    
    Theta measures time decay - how much value the option loses as one 
    day passes (assuming everything else stays constant).
    
    Theta is usually NEGATIVE for long options (you lose money over time).
    
    Formula:
        Call Theta = -[S·N'(d1)·σ / (2√T)] - r·K·e^(-rT)·N(d2)
        Put Theta = -[S·N'(d1)·σ / (2√T)] + r·K·e^(-rT)·N(-d2)
    
    Where:
        d1 = [ln(S/K) + (r + σ²/2)T] / (σ√T)
        d2 = d1 - σ√T
        N'(x) = standard normal PDF
    
    Args:
        S: Current stock price
        K: Strike price
        T: Time to expiration (years)
        r: Risk-free rate (annual)
        sigma: Volatility (annual)
        option_type: 'call' or 'put'
    
    Returns:
        Theta value (price change per day, typically negative)
    
    Examples:
        >>> theta(100, 100, 1.0, 0.05, 0.20, 'call')
        -0.0198...  # Loses ~$0.02 per day
        
        >>> theta(100, 100, 0.1, 0.05, 0.20, 'call')
        -0.0888...  # More negative near expiration
    
    Edge Cases:
        - Near expiration: Theta becomes very negative for ATM
        - Deep ITM puts: Theta can be POSITIVE (due to interest on strike)
        - At expiration: Theta is undefined
    
    Interview Question: "Why is theta negative?"
        Options are "wasting assets." As time passes, there's less time 
        for the stock to move, so the option's time value decays. This 
        accelerates near expiration (gamma risk).
    """
    # Theta is usually expressed per day, so divide annual theta by 365
    if T == 0:
        return None
    
    # Calculate d1 and d2
    denom = sigma* np.sqrt(T)
    d_1_num = np.log(S / K) + (r + (sigma ** 2) / 2) * T
    d1 = d_1_num / denom if denom != 0 else 0
    d2 = d1 - sigma * np.sqrt(T)

    # Calculate option value and cost
    op_val = S * norm.pdf(d1) * sigma / (2 * np.sqrt(T))
    op_cost = r * K * np.exp(-r * T)

    if option_type == 'call':
        return (-op_val - op_cost * norm.cdf(d2)) / 365
    if option_type == 'put':
        return (-op_val + op_cost * norm.cdf(-d2)) / 365



In [158]:
S, K, T, r, sigma, option_type = 317.32, 295, 13/365, 0.02, 0.3206, 'call'
print(f"Option Price: {black_scholes_price(S=S,K=K, T=T, r=r, sigma=sigma, option_type=option_type)}")
print(f"Option Delta: {delta(S=S,K=K, T=T, r=r, sigma=sigma, option_type=option_type)}")
print(f"Option Gamma: {gamma(S=S,K=K, T=T, r=r, sigma=sigma, option_type=option_type)}")
print(f"Option Vega: {vega(S=S,K=K, T=T, r=r, sigma=sigma, option_type=option_type)}")
print(f"Option Theta: {theta(S=S,K=K, T=T, r=r, sigma=sigma, option_type=option_type)}")
print(f"Option Rho: {rho(S=S,K=K, T=T, r=r, sigma=sigma, option_type=option_type)}")


Option Price: 23.53179221229732
Option Delta: 0.8938885136041462
Option Gamma: 0.009543332331240124
Option Vega: 0.10972584771832081
Option Theta: -0.14955339210703844
Option Rho: 0.09264437923340862


In [ ]:
S, K, T, r, sigma, option_type = 398.5400085449219, 280.0, 58/365, 0.023929998874664307, 0.16980747153018075, 'put'

In [288]:
delta(S, K, T, r, sigma, option_type)

np.float64(0.8938885136041462)

In [285]:
def black_scholes_price(
    S: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    option_type: Literal['call', 'put'] = 'call'
) -> float:
    """
    Calculate European option price using Black-Scholes model.
    
    Args:
        S: Current underlying price
        K: Strike price
        T: Time to expiration in years (e.g., 30 days = 30/365)
        r: Risk-free rate (annualized, as decimal, e.g., 0.02 for 2%)
        sigma: Implied volatility (annualized, as decimal, e.g., 0.20 for 20%)
        option_type: 'call' or 'put'
        
    Returns:
        Theoretical option price
        
    Example:
        >>> price = black_scholes_price(S=100, K=100, T=1.0, r=0.05, sigma=0.20)
        >>> print(f"Call price: ${price:.2f}")
        Call price: $10.45
        
    TODO: Implement Black-Scholes formula
    
    Hints:
        1. Handle edge case: T = 0 (expiration) → return intrinsic value
        2. Handle edge case: sigma = 0 → return discounted intrinsic value
        3. Calculate d1 = [ln(S/K) + (r + sigma²/2)*T] / (sigma*sqrt(T))
        4. Calculate d2 = d1 - sigma*sqrt(T)
        5. Use scipy.stats.norm.cdf() for N(d1) and N(d2)
        6. Calculate call = S*N(d1) - K*exp(-r*T)*N(d2)
        7. For put, use put-call parity or put formula directly
        
    Notes:
        - All inputs should be positive (except r can be negative in rare cases)
        - T is in YEARS (divide days by 365)
        - sigma and r are ANNUALIZED rates
    """
    # Input validation
    if S <= 0:
        raise ValueError(f"Underlying price must be positive, got {S}")
    if K <= 0:
        raise ValueError(f"Strike price must be positive, got {K}")
    if T < 0:
        raise ValueError(f"Time to expiration cannot be negative, got {T}")
    if sigma < 0:
        raise ValueError(f"Volatility cannot be negative, got {sigma}")
    if option_type not in ['call', 'put']:
        raise ValueError(f"option_type must be 'call' or 'put', got {option_type}")

    # Calculate d1 and d2
    denom = sigma* np.sqrt(T)
    d_1_num = np.log(S / K) + (r + (sigma ** 2) / 2) * T
    d1 = d_1_num / denom if denom != 0 else 0
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'call':
        # If contract expired, return 0 or difference between underlying and strike
        if T == 0:
            return intrinsic_value(S, K, option_type)
        
        # If no vol, return discounted intrinsic value
        if sigma == 0:
            return max(S - K * np.exp(-r*T), 0)

        return (S * norm.cdf(d1)) - (K * np.exp(-r*T) * norm.cdf(d2))

    if option_type == 'put':
        # If contract expired, return 0 or difference between underlying and strike
        if T == 0:
            return intrinsic_value(S, K, option_type)

        # If no vol, return discounted intrinsic value
        if sigma == 0:
            return max((K * np.exp(-r*T)) - S, 0)

        return (K * np.exp(-r*T) * norm.cdf(-d2)) - (S * norm.cdf(-d1)) 
from tqdm import tqdm
def implied_volatility(
    option_price: float,
    S: float,
    K: float,
    T: float,
    r: float,
    option_type: Literal['call', 'put'],
    initial_guess: float = 0.25,
    max_iterations: int = 10,
    tolerance: float = 1e-6
) -> float:
    """
    Calculate implied volatility using Newton-Raphson method.
    
    Implied volatility (IV) is the "market's opinion" of future volatility.
    It's the σ value that makes Black-Scholes price match the observed 
    market price.
    
    This is an inverse problem: we know the price, solve for σ.
    
    Newton-Raphson Formula:
        σ_new = σ_old - f(σ) / f'(σ)
    
    Where:
        f(σ) = BS_price(σ) - market_price
        f'(σ) = vega(σ)
    
    Args:
        option_price: Observed market price
        S: Current stock price
        K: Strike price
        T: Time to expiration (years)
        r: Risk-free rate (annual)
        option_type: 'call' or 'put'
        initial_guess: Starting volatility guess (default 25%)
        max_iterations: Maximum Newton-Raphson iterations
        tolerance: Convergence tolerance
    
    Returns:
        Implied volatility (annual)
     
    Raises:
        ValueError: If IV cannot be found (price violates arbitrage bounds)
    
    Examples:
        >>> # If market price is $10, what vol is implied?
        >>> implied_volatility(10.0, 100, 100, 1.0, 0.05, 'call')
        0.1987...  # ~19.87% implied vol
    
    Edge Cases:
        - Price below intrinsic value: Raise ValueError (arbitrage)
        - Very deep ITM/OTM: Vega → 0, Newton-Raphson fails
        - At expiration: IV is meaningless
    
    Interview Question: "Why solve for IV instead of using historical vol?"
        Historical vol tells you what WAS. Implied vol tells you what 
        the MARKET THINKS will happen. IV is forward-looking and 
        incorporates all market participants' views.
    
    Advanced: "What's the volatility smile?"
        In reality, IV varies by strike (smile/smirk). This violates 
        Black-Scholes assumptions. Far OTM puts have higher IV due to 
        crash risk. This is called "volatility skew."
    """
    # Check for arbitrage violations
    intrinsic = intrinsic_value(S, K, option_type)
    if option_price < intrinsic:
        raise ValueError(
            f"Option price ${option_price:.2f} below intrinsic ${intrinsic:.2f} "
            f"- arbitrage violation"
        )
    
    # Edge case: at expiration, IV is meaningless
    if T <= 0:
        raise ValueError("Cannot calculate IV at or past expiration")
    
    # Edge case: if price equals intrinsic, vol is effectively zero
    if abs(option_price - intrinsic) < 1e-6:
        return 0.001  # Minimum vol

    # 1. Start with initial guess (e.g., 25%)
    guess, n = initial_guess, 1

    while n < max_iterations:
        # 2. Calculate BS price with current guess
        bs_price_init = black_scholes_price(S, K, T, r, guess, option_type)
        err = bs_price_init - option_price

        # If BS price within tolerance bound of market price, return vol estimate
        if abs(err) < tolerance:
            return guess

        # Calculate function and derivate for Newton approximation recursion
        f_prime = vega(S, K, T, r, guess, option_type) * 100 # Scale vega  to per 100% volatility!
        print(f"Iteraion {n}, error: {err}, F': {f_prime}")
        update = err / f_prime 
        guess = guess - update

        # Clip guess to 'reasonable' volatility bounds
        guess = np.clip(guess, 0.001, 10.0)
        n += 1
    
    return guess

implied_volatility(22.35, S,K,T,r,option_type)

Iteraion 1, error: 0.5474820269589245, F': 6.807914799743446
Iteraion 2, error: 0.21577292681003968, F': 1.6303150414590115
Iteraion 3, error: 0.1800621607590287, F': 3.2027296286416384e-23
Iteraion 4, error: 0.1800621607590287, F': 0.0
Iteraion 5, error: 0.1800621607590287, F': 0.0
Iteraion 6, error: 0.1800621607590287, F': 0.0
Iteraion 7, error: 0.1800621607590287, F': 0.0
Iteraion 8, error: 0.1800621607590287, F': 0.0
Iteraion 9, error: 0.1800621607590287, F': 0.0


/tmp/ipykernel_70722/3502381449.py:179: RuntimeWarning: divide by zero encountered in scalar divide
  update = err / f_prime


np.float64(0.001)

In [ ]:
"strike":295,"Call","bid":22.44,"ask":22.86,"vol":0.3206,"delta":0.8706,"gamma":0.0103,"theta":-0.0788,"vega":0.1281,"rho":0.0973,
"mid_price":22.65,"days_to_expiry":13,"underlying_price":317.32000732421875,"risk_free_rate":0.02


In [181]:
black_scholes_price(S, K, T, r, 0.01, option_type)


np.float64(0.06807914799743446)

In [211]:
def calculate_realized_volatility(
    prices: pd.Series,
    window: int = 30,
    method: Literal['close', 'parkinson', 'garman_klass'] = 'close',
    annualize: bool = True
) -> pd.Series:
    """
    Calculate rolling realized volatility from price data.
    
    Args:
        prices: pandas Series of prices (typically 'close' prices)
        window: Rolling window size in days (default 30)
        method: Calculation method:
            - 'close': Standard close-to-close volatility
            - 'parkinson': Uses high-low range (more efficient)
            - 'garman_klass': Uses OHLC (most efficient)
        annualize: If True, multiply by sqrt(252) to annualize
    
    Returns:
        pandas Series of rolling volatility estimates
        
    Example:
        >>> prices = pd.Series([100, 101, 99, 102, 98])
        >>> rv = calculate_realized_volatility(prices, window=3)
        >>> print(rv)
        
    TODO: Implement realized volatility calculation
    
    Hints for 'close' method:
        1. Calculate log returns: ln(P_t / P_t-1)
        2. Calculate rolling standard deviation of returns
        3. Annualize: std * sqrt(252) for daily data
        4. Handle NaNs at the start (first 'window' values)
    
    Interview Question: "Why use log returns instead of simple returns?"
        Log returns are additive over time, symmetric, and more suitable
        for statistical analysis. Also, Black-Scholes assumes log-normal
        stock prices, which means log returns are normally distributed.
    """
    log_returns = np.log(1 + prices.pct_change())
    rolling_stdev = log_returns.rolling(window=window, min_periods=2).std()
    
    if annualize:
        rolling_stdev = rolling_stdev * np.sqrt(252)

    return rolling_stdev


calculate_realized_volatility(prices, annualize=True)

0          NaN
1          NaN
2     0.336199
3     0.398371
4     0.492590
5     0.462737
6     0.418878
7     0.403861
8     0.413087
9     0.447035
10    0.436272
11    0.416615
12    0.586754
13    0.831125
14    0.958851
15    0.977391
16    0.945075
17    0.918503
18    0.898330
19    0.885762
20    0.865436
21    0.844226
22    0.826790
23    0.813987
24    0.807033
25    0.792861
26    0.777460
27    0.814350
28    0.896248
29    0.951977
dtype: float64

In [220]:
from dateutil.relativedelta import relativedelta
from datetime import datetime

date1 = datetime(year=2020, month=1, day=1)
date2 = datetime(year=2020, month=1, day=10)

relativedelta(date2, date1).days / 365.0

0.024657534246575342

In [258]:
rates_df = pd.read_parquet('../data/processed/risk_free_rate.parquet')
rates_df['date'] = pd.to_datetime(rates_df['date'])

In [259]:
saturdays = pd.to_datetime(options_df['date'])

print(rates_df['date'][5])
print(saturdays[0])
print(rates_df['date'][5] == saturdays[0])

2019-06-08 00:00:00
2019-06-08 00:00:00
True


In [279]:
from scipy.optimize import newton

def implied_volatility(
    option_price: float,
    S: float,
    K: float,
    T: float,
    r: float,
    option_type: Literal['call', 'put'],
    initial_guess: float = 0.25,
    max_iterations: int = 100,
    tolerance: float = 1e-6
) -> float:
    """
    Calculate implied volatility using Newton-Raphson method.
    
    Implied volatility (IV) is the "market's opinion" of future volatility.
    It's the σ value that makes Black-Scholes price match the observed 
    market price.
    
    This is an inverse problem: we know the price, solve for σ.
    
    Newton-Raphson Formula:
        σ_new = σ_old - f(σ) / f'(σ)
    
    Where:
        f(σ) = BS_price(σ) - market_price
        f'(σ) = vega(σ)
    
    Args:
        option_price: Observed market price
        S: Current stock price
        K: Strike price
        T: Time to expiration (years)
        r: Risk-free rate (annual)
        option_type: 'call' or 'put'
        initial_guess: Starting volatility guess (default 25%)
        max_iterations: Maximum Newton-Raphson iterations
        tolerance: Convergence tolerance
    
    Returns:
        Implied volatility (annual)
     
    Raises:
        ValueError: If IV cannot be found (price violates arbitrage bounds)
    
    Examples:
        >>> # If market price is $10, what vol is implied?
        >>> implied_volatility(10.0, 100, 100, 1.0, 0.05, 'call')
        0.1987...  # ~19.87% implied vol
    
    Edge Cases:
        - Price below intrinsic value: Raise ValueError (arbitrage)
        - Very deep ITM/OTM: Vega → 0, Newton-Raphson fails
        - At expiration: IV is meaningless
    
    Interview Question: "Why solve for IV instead of using historical vol?"
        Historical vol tells you what WAS. Implied vol tells you what 
        the MARKET THINKS will happen. IV is forward-looking and 
        incorporates all market participants' views.
    
    Advanced: "What's the volatility smile?"
        In reality, IV varies by strike (smile/smirk). This violates 
        Black-Scholes assumptions. Far OTM puts have higher IV due to 
        crash risk. This is called "volatility skew."
    """
    
    # Validation: Check for arbitrage violations
    intrinsic = intrinsic_value(S, K, option_type)
    if option_price < intrinsic - 0.01:
        raise ValueError(
            f"Option price ${option_price:.2f} below intrinsic value ${intrinsic:.2f}. "
            f"This violates no-arbitrage"
        )
    
    # Edge case: at or past expiration
    if T <= 0:
        raise ValueError("Cannot calculate implied volatility at or past expiration")
    
    # Edge case: price equals intrinsic (zero time value)
    if abs(option_price - intrinsic) < 0.01:
        return 0.01  # Minimum vol (essentially zero time value)
    
    # Define the objective function: f(σ) = BS_price(σ) - market_price
    def objective(sigma):
        """Function to find root of (should equal zero at solution)."""
        sigma = np.clip(sigma, 0.0001, 5.0)  # ← Prevent invalid values

        return black_scholes_price(S, K, T, r, sigma, option_type) - option_price

    
    # Define the derivative: f'(σ) = vega(σ) 
    def derivative(sigma):
        """Derivative of objective function (vega)."""
        sigma = np.clip(sigma, 0.0001, 5.0)  # ← Prevent invalid values

        # Get vega (per 1% vol change)
        option_vega = vega(S, K, T, r, sigma, option_type)
        
        # Scale vega from per-1% to per-100% for Newton-Raphson
        scaled_vega = option_vega * 100
        
        # Prevent division by zero
        if abs(scaled_vega) < 1e-10:
            return 1e-10
        
        return scaled_vega

    
    try:
        # Use scipy's Newton-Raphson solver
        solved_iv = newton(
            func=objective,
            x0=initial_guess,
            fprime=derivative,
            tol=tolerance,
            maxiter=max_iterations,
            full_output=False
        )
        
        # Bound result to reasonable range
        solved_iv = np.clip(solved_iv, 0.001, 5.0)
        
        # Final verification: does this sigma actually produce the market price?
        verification_price = black_scholes_price(S, K, T, r, solved_iv, option_type)
        if abs(verification_price - option_price) > 0.10:
            raise ValueError(
                f"IV solution verification failed. "
                f"Solved IV={solved_iv:.4f} gives price ${verification_price:.2f}, "
                f"but market price is ${option_price:.2f}"
            )
        
        return solved_iv
        
    except RuntimeError as e:
        # Newton-Raphson failed to converge
        raise ValueError(
            f"IV solver failed to converge after {max_iterations} iterations. "
            f"Option characteristics: S=${S:.2f}, K=${K:.2f}, T={T:.4f}, "
            f"Price=${option_price:.2f}, Intrinsic=${intrinsic:.2f}, Type={option_type}"
        ) from e

implied_volatility(
    0.34,
    321.08,
    263.0,
    0.0027,
    0.02,
    'put'
)

ValueError: IV solver failed to converge after 100 iterations. Option characteristics: S=$321.08, K=$263.00, T=0.0027, Price=$0.34, Intrinsic=$0.00, Type=put

In [ ]:
Option characteristics: S=$321.86, K=$270.00, T=0.0274, Price=$0.15, Intrinsic=$0.00
Defaulting to 0.2 Vol

In [281]:
def implied_volatility(
    option_price: float,
    S: float,
    K: float,
    T: float,
    r: float,
    option_type: Literal['call', 'put'],
    initial_guess: float = 0.25,
    max_iterations: int = 100,
    tolerance: float = 1e-6
) -> float:
    """
    Calculate implied volatility using Newton-Raphson method with fallback.
    
    Args:
        option_price: Observed market price
        S: Current stock price
        K: Strike price
        T: Time to expiration (years)
        r: Risk-free rate (annual)
        option_type: 'call' or 'put'
        initial_guess: Starting volatility guess (default 25%)
        max_iterations: Maximum iterations
        tolerance: Convergence tolerance
    
    Returns:
        Implied volatility (annual)
     
    Raises:
        ValueError: If IV cannot be found (price violates arbitrage bounds)

    """
    
    # Validation: Check for arbitrage violations
    intrinsic = intrinsic_value(S, K, option_type)
    if option_price < intrinsic - 0.01:
        raise ValueError(
            f"Option price ${option_price:.2f} below intrinsic value ${intrinsic:.2f}. "
            f"This violates no-arbitrage"
            f"Option characteristics: S=${S:.2f}, K=${K:.2f}, T={T:.4f}, "
            f"Price=${option_price:.2f}, Intrinsic=${intrinsic:.2f}, Type={option_type}. "
            f"This may indicate an arbitrage violation or extreme market conditions."
        )
    
    # Edge case: at or past expiration
    if T <= 0:
        raise ValueError("Cannot calculate implied volatility at or past expiration")
    
    # Edge case: price equals intrinsic (zero time value)
    if abs(option_price - intrinsic) < 0.01:
        return 0.01  # Minimum vol (essentially zero time value)
    
    # For deep OTM options near expiry, use a higher initial guess
    # (These cases often have very high implied vol)
    if T < 0.05:  # Less than ~18 days to expiry
        if option_type == 'call' and S / K < 0.90:  # Deep OTM call
            initial_guess = min(1.0, initial_guess * 3)
        elif option_type == 'put' and S / K > 1.10:  # Deep OTM put
            initial_guess = min(1.0, initial_guess * 3)
    
    # Define the objective function: f(σ) = BS_price(σ) - market_price
    def objective(sigma):
        """Function to find root of (should equal zero at solution)."""
        sigma = np.clip(sigma, 0.0001, 5.0)  # ← Prevent invalid values

        return black_scholes_price(S, K, T, r, sigma, option_type) - option_price

    
    # Define the derivative: f'(σ) = vega(σ) 
    def derivative(sigma):
        """Derivative of objective function (vega)."""
        sigma = np.clip(sigma, 0.0001, 5.0)  # ← Prevent invalid values

        # Get vega (per 1% vol change)
        option_vega = vega(S, K, T, r, sigma, option_type)
        
        # Scale vega from per-1% to per-100% for Newton-Raphson
        scaled_vega = option_vega * 100
        
        # Prevent division by zero
        if abs(scaled_vega) < 1e-10:
            return 1e-10
        
        return scaled_vega

    
    try:
        # Try Newton-Raphson first (fast when it works)
        solved_iv = newton(
            func=objective,
            x0=initial_guess,
            fprime=derivative,
            tol=tolerance,
            maxiter=max_iterations,
            full_output=False
        )
        
        # Bound result to reasonable range
        solved_iv = np.clip(solved_iv, 0.001, 5.0)
        
        # Final verification: does this sigma actually produce the market price?
        verification_price = black_scholes_price(S, K, T, r, solved_iv, option_type)
        if abs(verification_price - option_price) > 0.10:
            raise ValueError(
                f"IV solution verification failed. "
                f"Solved IV={solved_iv:.4f} gives price ${verification_price:.2f}, "
                f"but market price is ${option_price:.2f}"
            )
        
        return solved_iv
        
    except RuntimeError:
        # Newton-Raphson failed - fallback to bounded bisection method
        # This is more robust for deep OTM options near expiry (low vega)
        try:
            # Use Brent's method with bounds [0.001, 5.0]
            solved_iv = brentq(
                f=objective,
                a=0.001,  # Lower bound on volatility
                b=5.0,    # Upper bound on volatility
                xtol=tolerance,
                maxiter=max_iterations
            )
            
            # Verify solution
            verification_price = black_scholes_price(S, K, T, r, solved_iv, option_type)
            if abs(verification_price - option_price) > 0.10:
                raise ValueError(
                    f"IV solution verification failed (Brent's method). "
                    f"Solved IV={solved_iv:.4f} gives price ${verification_price:.2f}, "
                    f"but market price is ${option_price:.2f}"
                )
            
            return solved_iv
            
        except ValueError as e:
            # Even bounded method failed - likely arbitrage violation or extreme case
            raise ValueError(
                f"IV solver failed (both Newton-Raphson and Brent's method). "
                f"Option characteristics: S=${S:.2f}, K=${K:.2f}, T={T:.4f}, "
                f"Price=${option_price:.2f}, Intrinsic=${intrinsic:.2f}, Type={option_type}. "
                f"This may indicate an arbitrage violation or extreme market conditions."
            ) from e



In [ ]:
 S=$320.73, K=$315.00, T=0.0384, Price=$3.27, Intrinsic=$5.73, Type=call.

In [282]:
implied_volatility(
    option_price=3.27,
    S=320.73,
    K=315,
    T=0.0384,
    r=0.02,
    option_type='call'

)

ValueError: Option price $3.27 below intrinsic value $5.73. This violates no-arbitrageOption characteristics: S=$320.73, K=$315.00, T=0.0384, Price=$3.27, Intrinsic=$5.73, Type=call. This may indicate an arbitrage violation or extreme market conditions.

In [290]:
options_df = pd.read_parquet('../data/processed/spy_options.parquet')


In [293]:
options_df['date'].unique()

array(['2023-12-05', '2023-12-06', '2023-12-07', '2023-12-08',
       '2023-12-11', '2023-12-12', '2023-12-13', '2023-12-14',
       '2023-12-15', '2023-12-18', '2023-12-19', '2023-12-20',
       '2023-12-21', '2023-12-22', '2023-12-26', '2023-12-27',
       '2023-12-28', '2023-12-29', '2024-01-02', '2024-01-03',
       '2024-01-04', '2024-01-05', '2024-01-08', '2024-01-09',
       '2024-01-10', '2024-01-11', '2024-01-12', '2024-01-16',
       '2024-01-17', '2024-01-18', '2024-01-19', '2024-01-22',
       '2024-01-23', '2024-01-24', '2024-01-25', '2024-01-26',
       '2024-01-29', '2024-01-30', '2024-01-31', '2024-02-01',
       '2024-02-02', '2024-02-05', '2024-02-06', '2024-02-07',
       '2024-02-08', '2024-02-09', '2024-02-12', '2024-02-13',
       '2024-02-14', '2024-02-15', '2024-02-16', '2024-02-20',
       '2024-02-21', '2024-02-22', '2024-02-23', '2024-02-26',
       '2024-02-27', '2024-02-28', '2024-02-29', '2024-03-01',
       '2024-03-04', '2024-03-05', '2024-03-06', '2024-